# 🚀 2D → 3D Reconstruction Pipeline (NVIDIA 3D Workflow)

**Hướng dẫn thao tác:** Chọn menu **Runtime ▸ Change runtime type ▸ T4 GPU** → sau đó chọn **Runtime ▸ Run all**.

---

### 🎯 Kiến trúc 2 Kịch bản Chuẩn hóa & 1 File Output Duy nhất:
- **Kịch bản 1: Đơn ảnh (1 ảnh duy nhất):**
  - Tiền xử lý P1 (Rembg, vá kín lỗ khúc xạ) → P2 `Depth-Anything-V2-Small` Pinhole Surface Mesh (~1.5s – 3.8s).
- **Kịch bản 2: Đa ảnh 360° (Từ 2 đến 8 ảnh):**
  - **P1 Preprocess:** Cân bằng màu DALI + Nhận diện mặt AI (CLIP ViT / HOG Bilateral Symmetry) + Gán cặp Hungarian $0^\circ, 90^\circ, 180^\circ, 270^\circ$.
  - **P2 Depth:** Dự đoán độ sâu đa góc + Lọc rách viền DA3-blender.
  - **P3 Quality Gate:** Kiểm định 3 tầng góc chụp & Fail-safe fallback.
  - **P4 TSDF Volumetric Mesh:** True Multi-View Silhouette Space Carving (triệt tiêu 100% phần dư) + Marching Cubes (kín nước Watertight, 0 cạnh hở) + Quadric Decimation.
  - **P5 Texture Blender:** Trải UV XAtlas siêu tốc + Phối màu Fresnel Angle-Weighted Blending ($\cos^3\theta$).
- **Output:** Luôn xuất ra đúng **1 file `.glb` duy nhất** tại `output/<tên_file>.glb`.

In [ ]:
# Cell 1: Clone Repository (nhánh P6-FullStack-Cloud) & Cài đặt thư viện
import os, sys, shutil

os.chdir('/content')
REPO = '/content/Img2d-to-3d'
BRANCH = 'P6-FullStack-Cloud'

if not os.path.isdir(REPO + '/.git'):
    shutil.rmtree(REPO, ignore_errors=True)
    !git clone -q --branch {BRANCH} https://github.com/dduy26/Img2d-to-3d.git {REPO}
else:
    !git -C {REPO} fetch -q origin
    !git -C {REPO} checkout -q {BRANCH}
    !git -C {REPO} reset --hard origin/{BRANCH}

os.chdir(REPO)
!git log --oneline -1

print('📦 Đang cài đặt các thư viện phụ thuộc...')
!pip install -q fastapi uvicorn python-multipart trimesh rembg onnxruntime networkx "scikit-image<0.26.0" opencv-python-headless transformers xatlas fast-simplification scipy

import trimesh
from transformers import AutoImageProcessor, AutoModelForDepthEstimation
print(f'✅ Thư viện sẵn sàng: trimesh {trimesh.__version__} | Depth-Anything-V2-Small OK | cwd: {os.getcwd()}')


In [ ]:
# Cell 2: Chuẩn bị ảnh đầu vào (Hỗ trợ 1 ảnh ĐƠN hoặc NHIỀU ảnh ĐA GÓC)
# - Bạn có thể upload ảnh của riêng bạn bằng nút Upload hoặc kéo thả vào thư mục input/ bên trái.
# - Nếu chưa có ảnh, hệ thống tự động nạp bộ 5 ảnh mẫu demo để kiểm thử ngay lập tức!

import os, glob, urllib.request
import matplotlib.pyplot as plt
from PIL import Image

os.chdir('/content/Img2d-to-3d')
INPUT_DIR = 'input'
os.makedirs(INPUT_DIR, exist_ok=True)

# Kiểm tra ảnh trong input/
existing_imgs = sorted([f for f in glob.glob(f'{INPUT_DIR}/*') if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')) and not f.endswith('.gitkeep')])

if len(existing_imgs) == 0:
    print('ℹ️ Thư mục input/ đang trống -> Đang tải bộ 5 ảnh chụp mẫu demo từ GitHub...')
    sample_base = 'https://raw.githubusercontent.com/dduy26/Img2d-to-3d/c41e16b/input'
    for i in range(1, 6):
        fname = f'view_0{i}.jpg'
        dst = os.path.join(INPUT_DIR, fname)
        try:
            urllib.request.urlretrieve(f'{sample_base}/{fname}', dst)
        except Exception as e:
            print(f'Lỗi tải {fname}:', e)
    existing_imgs = sorted([f for f in glob.glob(f'{INPUT_DIR}/*') if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp'))])

num_imgs = len(existing_imgs)
if num_imgs == 1:
    print(f'🎯 Phát hiện 1 ảnh: Kích hoạt chế độ ĐƠN ẢNH (Single-view Surface Mesh)')
elif num_imgs >= 2:
    print(f'🎯 Phát hiện {num_imgs} ảnh: Kích hoạt chế độ ĐA ẢNH 360° (NVIDIA Space Carving TSDF)')

# Hiển thị ảnh xem trước (Thumbnails)
cols = min(5, num_imgs)
plt.figure(figsize=(15, 3))
for idx, img_path in enumerate(existing_imgs[:5]):
    img = Image.open(img_path)
    plt.subplot(1, cols, idx + 1)
    plt.imshow(img)
    plt.title(os.path.basename(img_path), fontsize=10)
    plt.axis('off')
plt.tight_layout()
plt.show()
print('Danh sách ảnh trong input/:', [os.path.basename(p) for p in existing_imgs])


In [ ]:
# Cell 3: Khởi chạy FastAPI Server & Cloudflare Tunnel (Truy cập Web UI từ xa)
import subprocess, time, os, re, urllib.request, sys

REPO = '/content/Img2d-to-3d'
BACKEND = os.path.join(REPO, 'notebook', 'backend')
os.chdir(REPO)

# Dọn dẹp tiến trình cũ
!pkill -f uvicorn || true
!pkill -f cloudflared || true
time.sleep(2)

# Tải cloudflared nếu chưa có
if not os.path.exists('/usr/local/bin/cloudflared'):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared

env = dict(os.environ)
env['PYTHONPATH'] = BACKEND + os.pathsep + env.get('PYTHONPATH', '')

print('🚀 Đang khởi động FastAPI Backend (app.py)...')
server = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'app:app', '--host', '0.0.0.0', '--port', '8000'],
    cwd=BACKEND, env=env, stdout=open('/content/server.log', 'w'),
    stderr=subprocess.STDOUT, text=True,
)

# Chờ server sẵn sàng (Health check)
ready = False
for i in range(40):
    if server.poll() is not None:
        print('❌ Server thoát với mã lỗi:', server.returncode)
        break
    try:
        with urllib.request.urlopen('http://127.0.0.1:8000/api/health', timeout=2) as r:
            print(f'✅ Server READY ({i * 2}s):', r.read().decode())
            ready = True
            break
    except Exception:
        if i % 2 == 0 and i > 0:
            print(f'   ...đang nạp các mô hình ({i * 2}s)...')
        time.sleep(2)

# Bật Cloudflare Tunnel tạo link công khai
tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000', '--no-autoupdate'],
    stdout=open('/content/tunnel.log', 'w'), stderr=subprocess.STDOUT, text=True,
)

public_url = None
for _ in range(30):
    if os.path.exists('/content/tunnel.log'):
        log_text = open('/content/tunnel.log', encoding='utf-8', errors='replace').read()
        m = re.search(r'https://[\w.-]+\.trycloudflare\.com', log_text)
        if m:
            public_url = m.group(0)
            print('\n' + '=' * 60)
            print('🌐 LINK WEB UI (Xem 3D trực quan):', public_url)
            print('📘 LINK SWAGGER API DOCS        :', public_url + '/docs')
            print('=' * 60 + '\n')
            break
    time.sleep(1)


In [ ]:
# Cell 4: Chạy thử nghiệm trực tiếp Pipeline qua API Local
import glob, subprocess, json, os

os.chdir('/content/Img2d-to-3d')
imgs = sorted([f for f in glob.glob('input/*') if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp'))])
assert len(imgs) > 0, 'Chưa có ảnh trong input/ -> chạy lại Cell 2'

print(f'⚡ Đang thực thi tái tạo 3D với {len(imgs)} ảnh đầu vào...')
cmd = ['curl', '-s', '-X', 'POST', 'http://127.0.0.1:8000/generate-3d/']
for p in imgs:
    cmd += ['-F', f'files=@{p}']

r = subprocess.run(cmd, capture_output=True, text=True)

print('\n--- KẾT QUẢ TÁI TẠO 3D ---')
try:
    result_data = json.loads(r.stdout)
    print(json.dumps(result_data, indent=2, ensure_ascii=False))
    out_file = result_data.get('output_file')
    if out_file and os.path.exists(out_file):
        size_mb = os.path.getsize(out_file) / (1024 * 1024)
        print(f'\n✅ FILE 3D .GLB DUY NHẤT ĐÃ XUẤT: {out_file} ({size_mb:.2f} MB)')
except Exception:
    print('Lỗi phản hồi API:', r.stdout[:1000], r.stderr[-500:])


In [ ]:
# Cell 5: Trình xem 3D Interactive Three.js tích hợp trực tiếp trong Colab Notebook
import glob, os
from IPython.display import HTML, display

glbs = sorted(glob.glob('/content/Img2d-to-3d/output/*.glb'), key=os.path.getmtime, reverse=True)
if len(glbs) == 0:
    print('Chưa có file .glb nào trong output/.')
else:
    latest_glb = glbs[0]
    rel_path = os.path.basename(latest_glb)
    print(f'🎯 Đang hiển thị mô hình: {rel_path} ({os.path.getsize(latest_glb) / 1024:.1f} KB)')
    
    viewer_html = f'''
    <div style="width: 100%; height: 500px; background: #0f172a; border-radius: 12px; position: relative; overflow: hidden;">
      <div id="three-container" style="width: 100%; height: 100%;"></div>
      <div style="position: absolute; top: 12px; left: 16px; color: #38bdf8; font-family: sans-serif; font-size: 13px; font-weight: bold;">
        3D Model: {rel_path} (Kéo chuột xoay 360°, cuộn phóng to/thu nhỏ)
      </div>
    </div>
    <script type="module">
      import * as THREE from 'https://unpkg.com/three@0.160.0/build/three.module.js';
      import {{ OrbitControls }} from 'https://unpkg.com/three@0.160.0/examples/jsm/controls/OrbitControls.js';
      import {{ GLTFLoader }} from 'https://unpkg.com/three@0.160.0/examples/jsm/loaders/GLTFLoader.js';
      
      const container = document.getElementById('three-container');
      const scene = new THREE.Scene();
      scene.add(new THREE.HemisphereLight(0xffffff, 0x334155, 2.0));
      const dirLight = new THREE.DirectionalLight(0xffffff, 1.5);
      dirLight.position.set(3, 5, 4);
      scene.add(dirLight);
      
      const camera = new THREE.PerspectiveCamera(45, container.clientWidth / container.clientHeight, 0.01, 1000);
      const renderer = new THREE.WebGLRenderer({{ antialias: true, alpha: true }});
      renderer.setSize(container.clientWidth, container.clientHeight);
      container.appendChild(renderer.domElement);
      
      const controls = new OrbitControls(camera, renderer.domElement);
      controls.enableDamping = true;
      controls.autoRotate = true;
      controls.autoRotateSpeed = 2.0;
      
      const loader = new GLTFLoader();
      loader.load('http://127.0.0.1:8000/output/{rel_path}', (gltf) => {{
        const model = gltf.scene;
        model.traverse((o) => {{
          if (o.isMesh && o.material) {{
            if (o.geometry.attributes.color) o.material.vertexColors = true;
            o.material.side = THREE.DoubleSide;
            o.material.needsUpdate = true;
          }}
        }});
        const box = new THREE.Box3().setFromObject(model);
        const size = box.getSize(new THREE.Vector3()).length() || 1;
        const center = box.getCenter(new THREE.Vector3());
        model.position.sub(center);
        camera.position.set(0, size * 0.2, size * 1.3);
        scene.add(model);
      }});
      
      function animate() {{
        requestAnimationFrame(animate);
        controls.update();
        renderer.render(scene, camera);
      }}
      animate();
    </script>
    '''
    display(HTML(viewer_html))
